In [ ]:
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

municipalities = MunicipalityFeature("../../data/raw/municipalities_2022.csv")
municipalities = municipalities.load_transform()
municipalities

In [ ]:
from geoscore_de.data_flow.features.population import PopulationFeature

population = PopulationFeature(
    "../../data/raw/features/12411-02-03-5-population.csv", "../../data/tform/features/population.csv"
)
population = population.load_transform()
# merge population and municipalities on municipality ID
population = population.merge(municipalities, on="AGS", how="right")
population

In [ ]:
# difference between 2022 municipality data and 2024 population data
population["people_diff"] = population["total_population"] - population["Persons"]

In [ ]:
import plotnine as gg
from mizani.formatters import comma_format
from plotnine import aes, geom_histogram, ggplot, labs, theme_minimal

(
    ggplot(
        population.dropna(subset=["people_diff"]),
        aes(x="people_diff"),
    )
    + geom_histogram(bins=40, color="white")
    + labs(
        title="Distribution of Population Difference between 2022 municipality data and 2024 population data",
        x="Population difference (log scale)",
        y="Number of municipalities",
    )
    + gg.scale_x_log10(labels=comma_format())
    + theme_minimal()
)

Looks like basically every municipality increased in population

In [ ]:
import plotnine as gg
from mizani.formatters import comma_format
from plotnine import aes, geom_histogram, ggplot, labs, theme_minimal

(
    ggplot(
        population.dropna(subset=["total_population"]),
        aes(x="total_population"),
    )
    + geom_histogram(bins=40, color="white")
    + labs(
        title="Distribution of municipality size",
        x="Total population (log scale)",
        y="Number of municipalities",
    )
    + gg.scale_x_log10(labels=comma_format())
    + theme_minimal()
)

In [ ]:
import matplotlib.pyplot as plt

from geoscore_de.data_flow.geo import load_geo_data

age_share_groups = {
    "share_under_18": [
        "age_under_3",
        "age_3_to_5",
        "age_6_to_9",
        "age_10_to_14",
        "age_15_to_17",
    ],
    "share_working_age": [
        "age_18_to_19",
        "age_20_to_24",
        "age_25_to_29",
        "age_30_to_34",
        "age_35_to_39",
        "age_40_to_44",
        "age_45_to_49",
        "age_50_to_54",
        "age_55_to_59",
        "age_60_to_64",
    ],
    "share_65_plus": ["age_65_to_74", "age_75_and_over"],
}

population_map = population.copy()
for share_column, age_columns in age_share_groups.items():
    population_map[share_column] = population_map[age_columns].sum(axis=1, min_count=1)

geojson = load_geo_data("../../data/georef-germany-gemeinde.csv")

geojson_population = geojson.merge(
    population_map[["AGS", "share_under_18", "share_working_age", "share_65_plus"]],
    on="AGS",
    how="left",
)

fig, axes = plt.subplots(1, 3, figsize=(21, 8))
plot_specs = [
    ("share_under_18", "Proportion of Population under 18"),
    ("share_working_age", "Proportion of Working-age Population (18-64)"),
    ("share_65_plus", "Proportion of Population 65+"),
]

for ax, (column, title) in zip(axes, plot_specs):
    geojson_population.plot(
        column=column,
        ax=ax,
        cmap="YlOrRd",
        linewidth=0.05,
        edgecolor="white",
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        # vmin=0,
        # vmax=1,
        legend_kwds={"shrink": 0.7},
    )
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()
plt.show()